# pairadigm Example 

This notebook demonstrates how to use the `pairadigm` library to evaluate and score items based on pairwise comparisons. The example focuses on evaluating the valence of different text from the EMOBANK dataset. 

In [ ]:
# pip install pairadigm==0.5.1

In [19]:
import pandas as pd
import pairadigm as pdm
import dotenv
import os

In [ ]:
# NOTE: To re-run this example, you will need to replace this path with one to your own API keys. The APIs used here were OPENAI and ANTHROPIC. You will also need to download Ollama and install ministral-3:8b (or another model and change the name). Without these, you can at least see the example scoring, exploration, and evaluation below. 
 
# dotenv.load_dotenv(dotenv_path='YOUR PATH HERE')

In [22]:
# Load example data. Here we are using EMOBANK data.
data = pd.read_csv("data/emobank_small_sample.csv")
data

,id,split,V,A,D,text
0,SemEval_1155,train,3.36,3.00,3.27,Microsoft to release next generation phone
1,The_Black_Willow_11596_11663,train,2.70,3.00,2.80,"Allan crouched over his desk once more, pen in..."
2,113CWL018_952_1022,train,3.90,3.10,3.40,Your contribution last year of helped us get w...
3,detroit_12208_12263,train,3.10,3.00,3.10,"Did we build sane, sustainable, survival shelt..."
4,SemEval_763,train,2.86,3.00,2.86,Secret hotels in Irish countryside
5,SemEval_597,train,3.12,3.12,3.12,Stenson defends his title at Dubai
6,Anti-Terrorist_4031_4279,test,2.92,3.00,2.91,"Beyond these two sources, there is virtually n..."
7,116CUL034_796_978,train,3.20,3.00,3.50,We have met with a number of successes along t...
8,Ant_Robot_19422_19434,test,3.00,2.78,3.00,The Behavior
9,Nathans_Bylichka_15318_15334,train,3.56,3.00,3.22,"“Thanks, Dvorov."


In [23]:
# Load our CGCOT Prompts
with open('data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

In [24]:
# Initialize Pairadigm
p = pdm.Pairadigm(
    data = data,
    item_id_name = 'id',
    text_name = 'text',
    cgcot_prompts = val_prompts,
    model_name = ['gpt-5-nano', 'claude-haiku-4-5', 'ministral-3:8b'], 
    api_key = [os.getenv('OPENAI_API_KEY'), os.getenv('ANTHROPIC_API_KEY'), None],
    base_url = ["https://us.api.openai.com/v1", None, None],
    target_concept = 'valence'
)

In [25]:
# Review Clients
p.get_clients_info()

,index,model_name,provider
0,0,gpt-5-nano,openai
1,1,claude-haiku-4-5,anthropic
2,2,ministral-3:8b,ollama


In [26]:
# Get CGCOT Breakdowns
p.generate_breakdowns()


Generating breakdowns for 30 items using: gpt-5-nano
[30/30] 100.0% complete
Completed: 30/30 items

Generating breakdowns for 30 items using: claude-haiku-4-5
[30/30] 100.0% complete
Completed: 30/30 items

Generating breakdowns for 30 items using: ministral-3:8b
[30/30] 100.0% complete
Completed: 30/30 items

Breakdowns added to [object name].data with column name(s): CGCoT_Breakdown_gpt-5-nano, CGCoT_Breakdown_claude-haiku-4-5, CGCoT_Breakdown_ministral-3:8b


In [27]:
# Pair items
p.generate_pairings(breakdowns=True) # breakdowns=True let's the object know it should attach the generated breakdowns, otherwise it will just generate the pairs.

Pairwise DataFrame with breakdowns created and stored in self.pairwise_df


,item1,item2,breakdown1_gpt-5-nano,breakdown2_gpt-5-nano,breakdown1_claude-haiku-4-5,breakdown2_claude-haiku-4-5,breakdown1_ministral-3:8b,breakdown2_ministral-3:8b
0,116CUL034_796_978,Ant_Robot_19422_19434,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...
1,602CZL285_86_259,hotel-california_25973_25999,Original Text: Pat LaCrosse asked me to send t...,Original Text: No one else said anything.\nPro...,Original Text: Pat LaCrosse asked me to send t...,Original Text: No one else said anything.\nPro...,Original Text: Pat LaCrosse asked me to send t...,Original Text: No one else said anything.\nPro...
2,Nathans_Bylichka_29483_29506,116CUL034_2074_2422,"Original Text: “Roll up your sleeves,”\nPrompt...","Original Text: Since its re-organization, MCCO...","Original Text: “Roll up your sleeves,”\nPrompt...","Original Text: Since its re-organization, MCCO...","Original Text: “Roll up your sleeves,”\nPrompt...","Original Text: Since its re-organization, MCCO..."
3,Nathans_Bylichka_29483_29506,Nathans_Bylichka_38817_38837,"Original Text: “Roll up your sleeves,”\nPrompt...",Original Text: “Didn’t I warn you?”\nPrompt 1 ...,"Original Text: “Roll up your sleeves,”\nPrompt...",Original Text: “Didn’t I warn you?”\nPrompt 1 ...,"Original Text: “Roll up your sleeves,”\nPrompt...",Original Text: “Didn’t I warn you?”\nPrompt 1 ...
4,detroit_12208_12263,blog-new-year's-resolutions_5010_5074,"Original Text: Did we build sane, sustainable,...",Original Text: Self-efficacy did not have a si...,"Original Text: Did we build sane, sustainable,...",Original Text: Self-efficacy did not have a si...,"Original Text: Did we build sane, sustainable,...",Original Text: Self-efficacy did not have a si...
...,...,...,...,...,...,...,...,...
165,hotel-california_25973_25999,118CWL050_2436_2656,Original Text: No one else said anything.\nPro...,Original Text: Will you make a financial gift ...,Original Text: No one else said anything.\nPro...,Original Text: Will you make a financial gift ...,Original Text: No one else said anything.\nPro...,Original Text: Will you make a financial gift ...
166,captured_moments_33844_33911,captured_moments_5085_5254,Original Text: I heard no voices until Malaque...,Original Text: Her lower lip (a trifle too nar...,Original Text: I heard no voices until Malaque...,Original Text: Her lower lip (a trifle too nar...,Original Text: I heard no voices until Malaque...,Original Text: Her lower lip (a trifle too nar...
167,116CUL034_796_978,Anti-Terrorist_4031_4279,Original Text: We have met with a number of su...,"Original Text: Beyond these two sources, there...",Original Text: We have met with a number of su...,"Original Text: Beyond these two sources, there...",Original Text: We have met with a number of su...,"Original Text: Beyond these two sources, there..."
168,SemEval_763,blog-new-year's-resolutions_1573_1737,Original Text: Secret hotels in Irish countrys...,Original Text: An important premise to note he...,Original Text: Secret hotels in Irish countrys...,Original Text: An important premise to note he...,Original Text: Secret hotels in Irish countrys...,Original Text: An important premise to note he...


In [28]:
# Generate LLM Pairwise Annotations
p.generate_pairwise_annotations()

[gpt-5-nano] Completed 50/170 comparisons
[gpt-5-nano] Completed 100/170 comparisons
[gpt-5-nano] Completed 150/170 comparisons
[claude-haiku-4-5] Completed 50/170 comparisons
[claude-haiku-4-5] Completed 100/170 comparisons
[claude-haiku-4-5] Completed 150/170 comparisons
[ministral-3:8b] Completed 50/170 comparisons
[ministral-3:8b] Completed 100/170 comparisons
[ministral-3:8b] Completed 150/170 comparisons


,item1,item2,breakdown1_gpt-5-nano,breakdown2_gpt-5-nano,breakdown1_claude-haiku-4-5,breakdown2_claude-haiku-4-5,breakdown1_ministral-3:8b,breakdown2_ministral-3:8b,decision_gpt-5-nano,justification_gpt-5-nano,decision_claude-haiku-4-5,justification_claude-haiku-4-5,decision_ministral-3:8b,justification_ministral-3:8b
0,116CUL034_796_978,Ant_Robot_19422_19434,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Text1,"FINAL ANSWER: ""Description 1""\nJUSTIFICATION: ...",Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...,Text1,FINAL ANSWER: **Description 1**\n\nJUSTIFICATI...
1,602CZL285_86_259,hotel-california_25973_25999,Original Text: Pat LaCrosse asked me to send t...,Original Text: No one else said anything.\nPro...,Original Text: Pat LaCrosse asked me to send t...,Original Text: No one else said anything.\nPro...,Original Text: Pat LaCrosse asked me to send t...,Original Text: No one else said anything.\nPro...,Text1,FINAL ANSWER: Description 1\nJUSTIFICATION: It...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
2,Nathans_Bylichka_29483_29506,116CUL034_2074_2422,"Original Text: “Roll up your sleeves,”\nPrompt...","Original Text: Since its re-organization, MCCO...","Original Text: “Roll up your sleeves,”\nPrompt...","Original Text: Since its re-organization, MCCO...","Original Text: “Roll up your sleeves,”\nPrompt...","Original Text: Since its re-organization, MCCO...",Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
3,Nathans_Bylichka_29483_29506,Nathans_Bylichka_38817_38837,"Original Text: “Roll up your sleeves,”\nPrompt...",Original Text: “Didn’t I warn you?”\nPrompt 1 ...,"Original Text: “Roll up your sleeves,”\nPrompt...",Original Text: “Didn’t I warn you?”\nPrompt 1 ...,"Original Text: “Roll up your sleeves,”\nPrompt...",Original Text: “Didn’t I warn you?”\nPrompt 1 ...,Text1,FINAL ANSWER: Description 1\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...,Text1,FINAL ANSWER: **Description 1**\n\nJUSTIFICATI...
4,detroit_12208_12263,blog-new-year's-resolutions_5010_5074,"Original Text: Did we build sane, sustainable,...",Original Text: Self-efficacy did not have a si...,"Original Text: Did we build sane, sustainable,...",Original Text: Self-efficacy did not have a si...,"Original Text: Did we build sane, sustainable,...",Original Text: Self-efficacy did not have a si...,Text1,FINAL ANSWER: Description 1\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...,Text2,FINAL ANSWER: **Description 1**\n\nJUSTIFICATI...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,hotel-california_25973_25999,118CWL050_2436_2656,Original Text: No one else said anything.\nPro...,Original Text: Will you make a financial gift ...,Original Text: No one else said anything.\nPro...,Original Text: Will you make a financial gift ...,Original Text: No one else said anything.\nPro...,Original Text: Will you make a financial gift ...,Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
166,captured_moments_33844_33911,captured_moments_5085_5254,Original Text: I heard no voices until Malaque...,Original Text: Her lower lip (a trifle too nar...,Original Text: I heard no voices until Malaque...,Original Text: Her lower lip (a trifle too nar...,Original Text: I heard no voices until Malaque...,Original Text: Her lower lip (a trifle too nar...,Text1,FINAL ANSWER: Description 1\nJUSTIFICATION: De...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ..

NOTE: If you look at the decision_gpt-5-nano columns you'll see an example of a type of ERROR checking that pairadigm has built in: 

`ERROR from pairadigm (not model): Regex match NOT found even after recalling the model with an extraction prompt. Model response:` 

Since we do a regex check to look for the model's vote and can't guarantee it will follow instructions, we also use an extraction prompt to try to get the model to respond in the correct format. If it still doesn't match the regex, we return this error message. This is useful for debugging and understanding when models aren't following instructions properly. This has most notably been an issue with ChatGPT models.

In [31]:
p.irr()


INTER-RATER RELIABILITY RESULTS

LLM ANNOTATORS:
  Method: Krippendorff
  Score: 0.914
  Interpretation: Almost Perfect
  Annotators: 3
  Items: 170

ALL ANNOTATORS:
  Method: Krippendorff
  Score: 0.914
  Interpretation: Almost Perfect
  Annotators: 3
  Items: 170



,group,error,method,score,n_annotators,n_items,interpretation
0,llm,None,krippendorff,0.914031,3,170,Almost Perfect
1,all,None,krippendorff,0.914031,3,170,Almost Perfect


In [30]:
p.check_transitivity()

{'decision_gpt-5-nano': (0.96, 7, 175),
 'decision_claude-haiku-4-5': (0.9862385321100917, 3, 218),
 'decision_ministral-3:8b': (0.9403669724770642, 13, 218)}

In [34]:
# Since Claude had the highest transitivity, we'll use that model to score items.
p.score_items(decision_col='decision_claude-haiku-4-5', 
              normalization_scale='negative-one-to-one')

[claude-haiku-4-5] Bradley-Terry model fitted with 170 comparisons
[claude-haiku-4-5] Mean valence score: 0.127
[claude-haiku-4-5] Std valence score: 0.642
Score range: -1.000 to 1.000
25th percentile: -0.370
50th percentile (median): 0.268
75th percentile: 0.650

Highest scoring item on valence (score: 1.000):
We have met with a number of successes along the way, most notably the Summer Fun Line, the Metro Summer Bus Pass, and the development of ten neighborhood youth councils.

Lowest scoring item on valence (score: -1.000):
No one else said anything.
mean: 0.127
median: 0.268
std: 0.642
min: -1.000
max: 1.000
count: 30.000


,id,split,V,A,D,text,CGCoT_Breakdown_gpt-5-nano,CGCoT_Breakdown_claude-haiku-4-5,CGCoT_Breakdown_ministral-3:8b,Bradley_Terry_Score_claude-haiku-4-5
0,SemEval_1155,train,3.36,3.00,3.27,Microsoft to release next generation phone,Original Text: Microsoft to release next gener...,Original Text: Microsoft to release next gener...,Original Text: Microsoft to release next gener...,-0.098814
1,The_Black_Willow_11596_11663,train,2.70,3.00,2.80,"Allan crouched over his desk once more, pen in...",Original Text: Allan crouched over his desk on...,Original Text: Allan crouched over his desk on...,Original Text: Allan crouched over his desk on...,0.365281
2,113CWL018_952_1022,train,3.90,3.10,3.40,Your contribution last year of helped us get w...,Original Text: Your contribution last year of ...,Original Text: Your contribution last year of ...,Original Text: Your contribution last year of ...,0.921743
3,detroit_12208_12263,train,3.10,3.00,3.10,"Did we build sane, sustainable, survival shelt...","Original Text: Did we build sane, sustainable,...","Original Text: Did we build sane, sustainable,...","Original Text: Did we build sane, sustainable,...",0.062059
4,SemEval_763,train,2.86,3.00,2.86,Secret hotels in Irish countryside,Original Text: Secret hotels in Irish countrys...,Original Text: Secret hotels in Irish countrys...,Original Text: Secret hotels in Irish countrys...,0.858208
5,SemEval_597,train,3.12,3.12,3.12,Stenson defends his title at Dubai,Original Text: Stenson defends his title at Du...,Original Text: Stenson defends his title at Du...,Original Text: Stenson defends his title at Du...,0.465825
6,Anti-Terrorist_4031_4279,test,2.92,3.00,2.91,"Beyond these two sources, there is virtually n...","Original Text: Beyond these two sources, there...","Original Text: Beyond these two sources, there...","Original Text: Beyond these two sources, there...",0.307026
7,116CUL034_796_978,train,3.20,3.00,3.50,We have met with a number of successes along t...,Original Text: We have met with a number of su...,Original Text: We have met with a number of su...,Original Text: We have met with a number of su...,1.000000
8,Ant_Robot_19422_19434,test,3.00,2.78,3.00,The Behavior,Original Text: The Behavior\nPrompt 1 response...,Original Text: The Behavior\nPrompt 1 response...,Original Text: The Behavior\nPrompt 1 response...,-0.866151
9,Nathans_Bylichka_15318_15334,train,3.56,3.00,3.22,"“Thanks, Dvorov.","Original Text: “Thanks, Dvorov.\nPrompt 1 resp...","Original Text: “Thanks, Dvorov.\nPrompt 1 resp...","Original Text: “Thanks, Dvorov.\nPrompt 1 resp...",0.722379


In [39]:
p.plot_score_distribution('Bradley_Terry_Score_claude-haiku-4-5')

In [41]:
p.dawid_skene_annotator_ranking(random_seed=42)

Ranking 3 annotators across 170 instances...

DAWID-SKENE ANNOTATOR RANKING
Converged at iteration: 100

Top 5 Most Reliable Annotators:
   rank                  annotator  reliability type
0     1    decision_ministral-3:8b     0.169621  LLM
1     2        decision_gpt-5-nano     0.149161  LLM
2     3  decision_claude-haiku-4-5     0.129582  LLM




,annotator,reliability,type,rank
0,decision_ministral-3:8b,0.169621,LLM,1
1,decision_gpt-5-nano,0.149161,LLM,2
2,decision_claude-haiku-4-5,0.129582,LLM,3


In [42]:
p.save("data/valence_pairadigm_example.pkl")

Pairadigm object saved successfully to: data/valence_pairadigm_example.pkl
